## MERRA2 sub_surface analysis for all classes/groups

In [3]:
import os
import glob
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# 1. Paths
# -------------------------------------------------------------------
base_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean"
out_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/surf_layer_results"
os.makedirs(out_dir, exist_ok=True)

# -------------------------------------------------------------------
# 2. Utilities for incremental aggregation
# -------------------------------------------------------------------
def init_acc():
    # N, sum_x, sum_y, sum_diff, sum_diff2, sum_x2, sum_y2, sum_xy
    return dict(N=0, sum_x=0.0, sum_y=0.0,
                sum_diff=0.0, sum_diff2=0.0,
                sum_x2=0.0, sum_y2=0.0, sum_xy=0.0)

def update_acc(acc, x, y):
    """
    x, y are 1D numpy arrays (valid pairs only)
    Updates the accumulator in-place.
    """
    if len(x) == 0:
        return acc

    diff = x - y

    acc["N"] += len(x)
    acc["sum_x"] += x.sum()
    acc["sum_y"] += y.sum()
    acc["sum_diff"] += diff.sum()
    acc["sum_diff2"] += (diff ** 2).sum()
    acc["sum_x2"] += (x ** 2).sum()
    acc["sum_y2"] += (y ** 2).sum()
    acc["sum_xy"] += (x * y).sum()

    return acc

def finalize_metrics(acc):
    """
    From accumulator to bias, ubRMSD, R, N.
    Bias = mean(product - reference)
    ubRMSD removes the mean bias before RMSD.
    R is Pearson correlation via covariance formulation.
    """
    N = acc["N"]
    if N == 0:
        return np.nan, np.nan, np.nan, 0

    sum_x = acc["sum_x"]
    sum_y = acc["sum_y"]
    sum_diff = acc["sum_diff"]
    sum_diff2 = acc["sum_diff2"]
    sum_x2 = acc["sum_x2"]
    sum_y2 = acc["sum_y2"]
    sum_xy = acc["sum_xy"]

    # Mean diff (bias)
    mean_diff = sum_diff / N
    bias = mean_diff

    # E[(diff - mean_diff)^2] = E[diff^2] - mean_diff^2
    mean_diff2 = sum_diff2 / N
    var_ubrmsd = mean_diff2 - mean_diff ** 2
    if var_ubrmsd < 0:
        var_ubrmsd = 0.0
    ubrmsd = np.sqrt(var_ubrmsd)

    # Correlation R
    Ex = sum_x / N
    Ey = sum_y / N
    Ex2 = sum_x2 / N
    Ey2 = sum_y2 / N
    Exy = sum_xy / N

    var_x = Ex2 - Ex ** 2
    var_y = Ey2 - Ey ** 2

    if var_x <= 0 or var_y <= 0:
        R = np.nan
    else:
        cov_xy = Exy - Ex * Ey
        R = cov_xy / np.sqrt(var_x * var_y)

    return bias, ubrmsd, R, N

# -------------------------------------------------------------------
# 3. Main accumulator dictionaries per grouping
# -------------------------------------------------------------------
acc_temp = {}    # key: temp_class
acc_lc = {}      # key: land_cover_group
acc_climate = {} # key: climate_group
acc_elev = {}    # key: elevation_class

# -------------------------------------------------------------------
# 4. Process MERRA-2 files one by one
# -------------------------------------------------------------------
csv_paths = glob.glob(os.path.join(base_dir, "**", "*.csv"), recursive=True)
if not csv_paths:
    raise RuntimeError(f"No CSV files found under {base_dir}")

for i, csv_path in enumerate(csv_paths, start=1):
    try:
        df = pd.read_csv(csv_path, low_memory=False)
    except Exception as e:
        print(f"Failed to read {csv_path}: {e}")
        continue

    # Need ts_merra2 and ts_reference to compute metrics
    if "ts_merra2" not in df.columns or "ts_reference" not in df.columns:
        continue

    # Ensure numeric
    df["ts_merra2"] = pd.to_numeric(df["ts_merra2"], errors="coerce")
    df["ts_reference"] = pd.to_numeric(df["ts_reference"], errors="coerce")

    # Valid pairs
    x_all = df["ts_merra2"].values
    y_all = df["ts_reference"].values
    mask_valid = np.isfinite(x_all) & np.isfinite(y_all)
    if not mask_valid.any():
        continue

    df_valid = df.loc[mask_valid].copy()

    # ----------------- temperature class -----------------
    if "temp_class" in df_valid.columns:
        for cls, sub in df_valid.groupby("temp_class"):
            if cls == "ST-Unknown":
                continue
            x = sub["ts_merra2"].values
            y = sub["ts_reference"].values
            if cls not in acc_temp:
                acc_temp[cls] = init_acc()
            acc_temp[cls] = update_acc(acc_temp[cls], x, y)

    # ----------------- land cover group ------------------
    if "land_cover_group" in df_valid.columns:
        for lc, sub in df_valid.groupby("land_cover_group"):
            x = sub["ts_merra2"].values
            y = sub["ts_reference"].values
            if lc not in acc_lc:
                acc_lc[lc] = init_acc()
            acc_lc[lc] = update_acc(acc_lc[lc], x, y)

    # ----------------- climate group ---------------------
    if "climate_group" in df_valid.columns:
        for cg, sub in df_valid.groupby("climate_group"):
            x = sub["ts_merra2"].values
            y = sub["ts_reference"].values
            if cg not in acc_climate:
                acc_climate[cg] = init_acc()
            acc_climate[cg] = update_acc(acc_climate[cg], x, y)

    # ----------------- elevation class -------------------
    if "elevation_class" in df_valid.columns:
        for ec, sub in df_valid.groupby("elevation_class"):
            x = sub["ts_merra2"].values
            y = sub["ts_reference"].values
            if ec not in acc_elev:
                acc_elev[ec] = init_acc()
            acc_elev[ec] = update_acc(acc_elev[ec], x, y)

    if i % 50 == 0:
        print(f"Processed {i} files...")

print("Finished processing all MERRA-2 files.")

# -------------------------------------------------------------------
# 5. Convert accumulators to DataFrames and save
# -------------------------------------------------------------------
def acc_to_df(acc_dict, group_name):
    rows = []
    for g, acc in sorted(acc_dict.items(), key=lambda kv: str(kv[0])):
        bias, ubrmsd, R, N = finalize_metrics(acc)
        rows.append(dict(
            group=g,
            bias=bias,
            ubrmsd=ubrmsd,
            R=R,
            N=N
        ))
    df_out = pd.DataFrame(rows)
    if not df_out.empty:
        df_out["bias"] = df_out["bias"].round(2)
        df_out["ubrmsd"] = df_out["ubrmsd"].round(2)
        df_out["R"] = df_out["R"].round(3)
    df_out.rename(columns={"group": group_name}, inplace=True)
    return df_out

# 5.1 Temperature classes
temp_stats = acc_to_df(acc_temp, "temp_class")
temp_path = os.path.join(out_dir, "merra2_L_234_metrics_by_temp_class.csv")
temp_stats.to_csv(temp_path, index=False)
print(f"Saved temperature-class metrics to: {temp_path}")

# 5.2 Land-cover groups
lc_stats = acc_to_df(acc_lc, "land_cover_group")
lc_path = os.path.join(out_dir, "merra2_L_234_metrics_by_land_cover_group.csv")
lc_stats.to_csv(lc_path, index=False)
print(f"Saved land-cover metrics to: {lc_path}")

# 5.3 Climate groups
climate_stats = acc_to_df(acc_climate, "climate_group")
climate_path = os.path.join(out_dir, "merra2_L_234_metrics_by_climate_group.csv")
climate_stats.to_csv(climate_path, index=False)
print(f"Saved climate metrics to: {climate_path}")

# 5.4 Elevation classes
elev_stats = acc_to_df(acc_elev, "elevation_class")
elev_path = os.path.join(out_dir, "merra2_L_234_metrics_by_elevation_class.csv")
elev_stats.to_csv(elev_path, index=False)
print(f"Saved elevation metrics to: {elev_path}")

# Optional quick preview
if not temp_stats.empty:
    print("\nMERRA-2 temperature-class metrics (preview):")
    print(temp_stats.to_markdown(index=False))


Processed 50 files...
Processed 100 files...
Processed 150 files...
Processed 200 files...
Processed 250 files...
Processed 300 files...
Processed 350 files...
Processed 400 files...
Processed 450 files...
Processed 500 files...
Processed 550 files...
Processed 600 files...
Finished processing all MERRA-2 files.
Saved temperature-class metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/surf_layer_results/merra2_L_234_metrics_by_temp_class.csv
Saved land-cover metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/surf_layer_results/merra2_L_234_metrics_by_land_cover_group.csv
Saved climate metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/surf_layer_results/merra2_L_234_metrics_by_climate_group.csv
Saved elevation metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/surf_layer_results/merra2_L_234_metrics_by_elevation_class.csv

MERRA-2 temperature-class me